# 11 — End-to-End Inference Pipeline

**Goal:** Wire together the NER model (notebook 5), the classifier (notebook 8), and the chunking logic (notebook 9) into a single function:

```python
analyze_report(text) -> {
    "category": str,
    "category_scores": dict,
    "entities": [ {text, type, score}, ... ]
}
```

This is the shape of a deployable inference service. Everything before was learning; this is the deliverable.

## Step 1 — Load both models once

In [1]:
import torch
import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    pipeline,
)

NER_DIR = "./ner-cti-bert-final"
CLS_DIR = "./cls-cti-bert-final"

# NER: use the HF pipeline with built-in entity aggregation
ner_pipe = pipeline(
    "token-classification",
    model=NER_DIR,
    tokenizer=NER_DIR,
    aggregation_strategy="simple",
)

# Classification: load raw so we can chunk long inputs ourselves
cls_tok = AutoTokenizer.from_pretrained(CLS_DIR)
cls_model = AutoModelForSequenceClassification.from_pretrained(CLS_DIR)
cls_model.eval()
cls_id2label = cls_model.config.id2label

Device set to use cuda:0


## Step 2 — Classification with chunking

In [2]:
def classify_report(text, max_length=512, stride=64):
    enc = cls_tok(
        text,
        max_length=max_length,
        stride=stride,
        truncation=True,
        return_overflowing_tokens=True,
        padding=True,
        return_tensors="pt",
    )
    with torch.no_grad():
        logits = cls_model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).logits
    probs = F.softmax(logits, dim=-1).mean(dim=0)         # average across chunks
    top = probs.argmax().item()
    return {
        "category": cls_id2label[top],
        "category_scores": {cls_id2label[i]: round(p.item(), 4) for i, p in enumerate(probs)},
    }

## Step 3 — NER with chunking

For long reports, we split into overlapping chunks, run NER on each, then dedupe by `(entity_text, entity_type)` across the overlap.

In [3]:
def chunk_by_chars(text, chunk_size=2000, overlap=200):
    """Simple character-level chunking — fine for NER in practice since the
    pipeline will re-tokenize internally and respect its own 512-token limit."""
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start : start + chunk_size])
        if start + chunk_size >= len(text):
            break
        start += chunk_size - overlap
    return chunks

def extract_entities(text):
    seen = set()
    entities = []
    for chunk in chunk_by_chars(text):
        for ent in ner_pipe(chunk):
            key = (ent["word"].strip().lower(), ent["entity_group"])
            if key in seen:
                continue
            seen.add(key)
            entities.append({
                "text": ent["word"],
                "type": ent["entity_group"],
                "score": round(float(ent["score"]), 4),
            })
    return entities

## Step 4 — The unified function

In [4]:
def analyze_report(text):
    cat = classify_report(text)
    return {
        "category": cat["category"],
        "category_scores": cat["category_scores"],
        "entities": extract_entities(text),
    }

## Step 5 — Try it

In [5]:
import json

report = """APT29 conducted a months-long espionage campaign against a European foreign ministry.
Initial access was achieved through spear-phishing emails carrying ISO attachments.
The attackers deployed a custom loader that fetched Cobalt Strike beacons from legitimate cloud storage.
Credential harvesting used Mimikatz, and persistence was established through scheduled tasks.
The group exploited CVE-2023-23397 where possible to bypass authentication.
Lateral movement reached the ministry's internal SharePoint where sensitive documents were exfiltrated.
"""

result = analyze_report(report)
print(json.dumps(result, indent=2))

{
  "category": "apt-campaign",
  "category_scores": {
    "apt-campaign": 0.3132,
    "phishing": 0.1002,
    "ransomware": 0.2162,
    "supply-chain": 0.1468,
    "vulnerability-report": 0.2237
  },
  "entities": [
    {
      "text": "apt",
      "type": "THREAT_ACTOR",
      "score": 0.2791
    },
    {
      "text": "cv",
      "type": "VULNERABILITY",
      "score": 0.4496
    }
  ]
}


## Step 6 — Performance & deployment notes

- **Batching**: if you have many reports, batch them. `ner_pipe([r1, r2, ...])` is much faster than calling once per report.
- **GPU**: move models to CUDA (`model.to("cuda")`; pipelines accept `device=0`) — huge speedup.
- **Warmup**: first call is slow because weights load lazily; keep models in memory for a long-running service.
- **Confidence thresholds**: filter NER output by `score >= 0.8` to trade recall for precision.
- **Serving**: wrap `analyze_report` in a FastAPI endpoint; use `uvicorn` for production.

## Your turn — final exercises

1. **Batch mode**: write `analyze_reports(texts: List[str])` that batches both NER and classification calls for a list of reports.
2. **Confidence threshold**: add a `min_score` parameter that drops low-confidence entities. What threshold maximizes F1 on a held-out gold set?
3. **Ship it**: wrap `analyze_report` in a FastAPI app with a `POST /analyze` endpoint. Test it with `curl`.
4. **Swap backbones**: repeat the whole pipeline with SecureBERT-trained models. Measure end-to-end quality difference.

## You made it

You now have:
- Conceptual fluency: tokens, [CLS], BIO, subword alignment, chunking
- Two fine-tuned CTI models: NER and classification
- A deployable inference function
- A template you can repeat on new datasets and entity types

Where to go next:
- **Larger datasets**: DNRTI for NER, your own collected reports for classification
- **Multi-label classification**: reports often fit multiple categories
- **MITRE ATT&CK mapping**: frame the task as classifying into tactics/techniques
- **Active learning**: use model uncertainty to prioritize which new reports to label
- **Larger / long-context models**: ModernBERT, Longformer when document length hurts